# 🎯 5. Advanced Indexing

Στο notebook 01 χρησιμοποιήσαμε ένα απλό μοτίβο: **κόψε σε chunks → embed → store**. Σε σοβαρά συστήματα αυτό συχνά **δεν φτάνει**. Σε αυτό το notebook θα δούμε τέσσερις προηγμένες στρατηγικές:

1. **Multi-Representation Indexing** — embed σύνοψη, return ολόκληρο doc
2. **Parent Document Retriever** — embed μικρά chunks, return τα parent
3. **RAPTOR** — ιεραρχικό clustering και summarization
4. **ColBERT (late interaction)** — fine-grained token-level matching
5. **Knowledge Graphs** — όταν τα facts είναι δομημένα

Όλες προσπαθούν να λύσουν το **κεντρικό dilemma**: μικρά chunks → καλύτερα embeddings, αλλά λιγότερο context. Μεγάλα chunks → καλύτερο context, αλλά αραιωμένα embeddings.

<img src="images/advanced_indexing_strategies.png" width="60%" style="border-radius:10px;margin:12px 0;"/>

***Εικ. 5.1** — Τρεις στρατηγικές advanced indexing. Κοινή αρχή: «Search what you embed, return what the LLM needs.»*

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(__file__).parent / ".env" if "__file__" in dir() else Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

In [2]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

embedder = OpenAIEmbeddings(model=EMBED_MODEL)
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

# Datanous.ai knowledge base — reused throughout this notebook
datanous_docs = [
    Document(page_content="Datanous.ai is an AI services company founded in 2021 in Athens, Greece, with offices in London and Amsterdam.",
             metadata={"source": "datanous_overview.txt", "product": "company"}),
    Document(page_content="Datanous Insight is a RAG-powered knowledge management platform that indexes enterprise documents and answers queries with source citations.",
             metadata={"source": "datanous_products.txt", "product": "Insight"}),
    Document(page_content="Datanous Insight Starter costs 50 USD per month and supports up to 10,000 document pages with single-tenant isolation.",
             metadata={"source": "datanous_products.txt", "product": "Insight", "section": "pricing"}),
    Document(page_content="Datanous Insight Professional costs 350 USD per month, supports up to 100,000 pages, and includes multi-tenant row-level access control.",
             metadata={"source": "datanous_products.txt", "product": "Insight", "section": "pricing"}),
    Document(page_content="Datanous Insight Enterprise offers unlimited pages, a dedicated vector store, and a 99.9 percent uptime SLA at custom pricing.",
             metadata={"source": "datanous_products.txt", "product": "Insight", "section": "pricing"}),
    Document(page_content="Datanous Search combines BM25 lexical retrieval with dense semantic embeddings and applies cross-encoder reranking via Reciprocal Rank Fusion. Latency under 200 ms.",
             metadata={"source": "datanous_products.txt", "product": "Search"}),
    Document(page_content="Datanous Guard enforces factual grounding, detects prompt injection attacks, redacts PII, and applies tenant isolation via tenant_id filtering.",
             metadata={"source": "datanous_products.txt", "product": "Guard"}),
    Document(page_content="Datanous Embed supports text-embedding-3-small (1536 dims) and text-embedding-3-large (3072 dims). Batch requests up to 2048 texts. Redis cache hit rate exceeds 60 percent.",
             metadata={"source": "datanous_products.txt", "product": "Embed"}),
    Document(page_content="Datanous.ai serves clients in banking, legal, healthcare, and media across Europe. All data encrypted with TLS 1.3 in transit and AES-256 at rest. GDPR-compliant.",
             metadata={"source": "datanous_faq.txt", "section": "privacy"}),
]
print(f"Setup complete! Instantiated {len(datanous_docs)} documents, LLM, and embedder.")

Setup complete! Instantiated 9 documents, LLM, and embedder.


## 5.1 Multi-Representation Indexing

**Ιδέα:** Φτιάχνουμε **δύο διαφορετικές αναπαραστάσεις** του ίδιου doc:
* Μια **σύντομη σύνοψη** που χρησιμοποιείται για **search** (αυτή κάνει embed)
* Το **πλήρες doc** που επιστρέφεται ως **context** στο LLM

Γιατί δουλεύει: η σύνοψη συμπυκνώνει το νόημα → καθαρότερο embedding, ενώ το LLM παίρνει όλη την πληροφορία.

<img src="images/multi-repr.png" width="60%" style="border-radius:10px;margin:12px 0;"/>

In [3]:
long_docs = [
    Document(page_content="""\
Datanous.ai Chapter 1 — The Limits of LLM Knowledge
An LLM is trained on a snapshot of text ending at a knowledge cutoff date.
Parametric knowledge is static, lossy, and unverifiable. Four limits drive RAG:
(1) knowledge cutoff, (2) private data invisibility, (3) hallucination,
(4) no live tools. Hallucination is fluent, plausible, and false — a direct
consequence of next-token prediction optimised for probability, not truth."""),
    Document(page_content="""\
Datanous.ai Chapter 2 — The Architecture of RAG
RAG is an architectural pattern: retrieve relevant chunks, inject into prompt,
generate grounded answer. Eight-stage pipeline: ingest → chunk → embed → store
(indexing); encode query → retrieve top-k → inject context → generate (query).
The vector store bridges both phases. Same embedding model must be used for
indexing and querying. Changing it requires re-embedding the entire corpus."""),
    Document(page_content="""\
Datanous.ai Chapter 3 — Embeddings and Vector Databases
Embedding models map text to fixed-dimension vectors; cosine similarity = cos(θ).
ANN indexes (HNSW, IVF) return approximate neighbours in milliseconds at scale.
Hybrid search combines BM25 (sparse/lexical) with dense embeddings via RRF.
BM25 excels at exact keyword and ID matching; dense excels at semantic synonyms.
Dimensionality trade-off: higher dimensions capture more nuance but cost more."""),
]

In [4]:
import uuid
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
 
# 1. Create a vector store for the summaries
vectorstore = Chroma(collection_name="summaries", embedding_function=embedder)
 
# 2. Create an in-memory store for the parent documents
store = InMemoryStore()
id_key = "doc_id"
 
# 3. The Retriever combines both
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)
 
# 4. Generate summaries for the long docs
chain = (
    ChatPromptTemplate.from_template("Summarize the following document in 1-2 precise sentences:\n\n{doc}")
    | llm
    | StrOutputParser()
)
 
doc_ids = [str(uuid.uuid4()) for _ in long_docs]
summary_docs = []
 
print("Generating summaries...")
for i, doc in enumerate(long_docs):
    summary = chain.invoke({"doc": doc.page_content})
    # Create a new document for the summary, containing the id of the original doc
    summary_doc = Document(page_content=summary, metadata={id_key: doc_ids[i]})
    summary_docs.append(summary_doc)
    print(f"\nDoc {i+1} Summary: {summary}")
 
# 5. Add summaries to vector store and original docs to doc store
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, long_docs)))
 
print("\nIndexing complete! Summaries are in Chroma, full docs are in InMemoryStore.")

Generating summaries...

Doc 1 Summary: Chapter 1 of Datanous.ai discusses the limitations of large language models (LLMs), highlighting that their knowledge is constrained by a cutoff date, lacks access to private data, can produce false information (hallucinations), and does not utilize live tools. These factors contribute to the static and unverifiable nature of parametric knowledge in LLMs.

Doc 2 Summary: Chapter 2 of Datanous.ai outlines the RAG architectural pattern, which involves an eight-stage pipeline for retrieving relevant information and generating grounded answers. It emphasizes the importance of using the same embedding model for both indexing and querying, as any changes necessitate re-embedding the entire corpus.

Doc 3 Summary: Chapter 3 of Datanous.ai discusses how embedding models convert text into fixed-dimension vectors and utilize approximate nearest neighbor (ANN) indexing methods like HNSW and IVF for efficient retrieval. It highlights the benefits of hybrid s

In [5]:
from IPython.display import display, Markdown
 
# Let's test the retriever
query = "What is hallucination according to Datanous?"
display(Markdown(f"Query: {query}\n"))
 
# 1. Find the best matching summary directly from the vectorstore
matched_summaries = vectorstore.similarity_search(query, k=1)
display(Markdown(f"Matched Summary in VectorStore:\n{matched_summaries[0].page_content}\n"))
display(Markdown(f"Matched Doc ID: {matched_summaries[0].metadata[id_key]}\n"))
 
# 2. The MultiVectorRetriever automatically fetches the parent document using the doc_id
retrieved_docs = retriever.invoke(query)
display(Markdown(f"Returned Parent Document by MultiVectorRetriever:\n{retrieved_docs[0].page_content}"))

Query: What is hallucination according to Datanous?


Matched Summary in VectorStore:
Chapter 1 of Datanous.ai discusses the limitations of large language models (LLMs), highlighting that their knowledge is constrained by a cutoff date, lacks access to private data, can produce false information (hallucinations), and does not utilize live tools. These factors contribute to the static and unverifiable nature of parametric knowledge in LLMs.


Matched Doc ID: 1adf5c58-365b-4700-bd6b-aadb28dd0ba0


Returned Parent Document by MultiVectorRetriever:
Datanous.ai Chapter 1 — The Limits of LLM Knowledge
An LLM is trained on a snapshot of text ending at a knowledge cutoff date.
Parametric knowledge is static, lossy, and unverifiable. Four limits drive RAG:
(1) knowledge cutoff, (2) private data invisibility, (3) hallucination,
(4) no live tools. Hallucination is fluent, plausible, and false — a direct
consequence of next-token prediction optimised for probability, not truth.

**Πότε ταιριάζει:** Όταν τα docs σου είναι **εκτενή** (τεχνικά manuals, legal contracts, research papers). Η σύνοψη γίνεται «καθαρός δείκτης», ενώ το LLM παίρνει όλη τη λεπτομέρεια.

## 5.2 Parent Document Retriever

**Ιδέα:** Έμμεση λύση στο ίδιο πρόβλημα.
* Σπάμε κάθε doc σε **πολλά μικρά child chunks** (πχ 100 tokens)
* Κάνουμε embed τα **child chunks** (έχουν εστιασμένο νόημα)
* Όταν matchάρει ένα child chunk, επιστρέφουμε το **parent** (μεγάλο chunk ή ολόκληρο doc)

Αυτό είναι παραλλαγή του multi-representation — αντί για summary, χρησιμοποιείς child chunks ως δείκτες.

In [6]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore
from langchain_chroma import Chroma
 
# 1. Define splitters for parent and child
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
 
# 2. Set up vector store and doc store
child_vectorstore = Chroma(collection_name="split_parents", embedding_function=embedder)
parent_store = InMemoryStore()
 
# 3. Create the retriever
parent_retriever = ParentDocumentRetriever(
    vectorstore=child_vectorstore,
    docstore=parent_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
 
# 4. Add the documents (this automatically splits them and creates the links)
parent_retriever.add_documents(datanous_docs)
print(f"Added {len(datanous_docs)} parent documents.")
print(f"Created {child_vectorstore._collection.count()} child chunks in vector store.")
 
# 5. Test it
query = "How much does the Starter plan cost?"
child_docs = child_vectorstore.similarity_search(query, k=1)
print(f"\nQuery: {query}\n")
print(f"Matched Child Chunk (from vector store):\n{child_docs[0].page_content}\n")
 
parent_docs = parent_retriever.invoke(query)
print(f"Returned Parent Document (from doc store):\n{parent_docs[0].page_content}")

Added 9 parent documents.
Created 18 child chunks in vector store.

Query: How much does the Starter plan cost?

Matched Child Chunk (from vector store):
Datanous Insight Starter costs 50 USD per month and supports up to 10,000 document pages with

Returned Parent Document (from doc store):
Datanous Insight Starter costs 50 USD per month and supports up to 10,000 document pages with single-tenant isolation.


## 5.3 RAPTOR — Ιεραρχικό clustering & summarization

Το **RAPTOR** (Recursive Abstractive Processing for Tree-Organized Retrieval) είναι από τις πιο εντυπωσιακές πρόσφατες ιδέες. 

**Ιδέα:**
1. Σπάμε όλο το corpus σε chunks (leaves)
2. Τα κάνουμε cluster (πχ με Gaussian Mixture)
3. Κάνουμε **summary** του κάθε cluster με LLM
4. Τα summaries γίνονται **νέα leaves** σε ψηλότερο επίπεδο
5. Επαναλαμβάνουμε μέχρι να φτάσουμε σε root
6. **Όλα** (leaves + intermediate summaries + root) γίνονται indexed

Αποτέλεσμα: για **specific** queries βρίσκει chunks, για **broad** queries βρίσκει high-level summaries.

<img src="images/raptor.png" width="60%" style="border-radius:10px;margin:12px 0;"/>

Παρακάτω παρουσιάζουμε μια **απλοποιημένη** εκδοχή για να καταλάβεις τη μηχανική (όχι production-grade).

In [7]:
# RAPTOR uses a corpus of text passages that it clusters and summarises hierarchically.
# Here we use Datanous.ai product and company content as the source corpus.
 
raptor_corpus = [
    "Datanous.ai is an AI services company founded in 2021 in Athens, Greece, with 85 engineers.",
    "Datanous Insight is a RAG-powered platform that indexes enterprise documents with source citations.",
    "Datanous Insight Starter costs 50 USD per month and supports up to 10,000 document pages.",
    "Datanous Insight Professional costs 350 USD per month and supports up to 100,000 pages.",
    "Datanous Insight Enterprise offers unlimited pages with a 99.9 percent uptime SLA.",
    "Datanous Search combines BM25 and dense embeddings via Reciprocal Rank Fusion. Latency under 200 ms.",
    "Datanous Guard performs factual grounding, prompt injection detection, PII redaction, and tenant isolation.",
    "Datanous Embed supports text-embedding-3-small (1536 dims) and text-embedding-3-large (3072 dims).",
    "Redis caching in Datanous Embed achieves a cache hit rate above 60 percent on enterprise corpora.",
    "All Datanous.ai deployments use TLS 1.3 in transit and AES-256 at rest. GDPR-compliant.",
    "A law firm reduced document search time from 2.5 hours to 8 minutes using Datanous Insight.",
    "A retail bank reduced regulatory query resolution from 4 days to 30 seconds with Datanous Insight.",
    "A hospital network reduced protocol retrieval from 6 minutes to 15 seconds with Datanous Insight.",
    "Datanous.ai evaluates faithfulness, answer relevancy, context precision, and context recall monthly.",
    "Datanous.ai is a certified partner of OpenAI, Anthropic, Pinecone, and Weaviate.",
]
 
print(f"RAPTOR corpus: {len(raptor_corpus)} passages")
for i, p in enumerate(raptor_corpus):
    print(f"  [{i+1:02d}] {p[:90]}")

RAPTOR corpus: 15 passages
  [01] Datanous.ai is an AI services company founded in 2021 in Athens, Greece, with 85 engineers
  [02] Datanous Insight is a RAG-powered platform that indexes enterprise documents with source c
  [03] Datanous Insight Starter costs 50 USD per month and supports up to 10,000 document pages.
  [04] Datanous Insight Professional costs 350 USD per month and supports up to 100,000 pages.
  [05] Datanous Insight Enterprise offers unlimited pages with a 99.9 percent uptime SLA.
  [06] Datanous Search combines BM25 and dense embeddings via Reciprocal Rank Fusion. Latency und
  [07] Datanous Guard performs factual grounding, prompt injection detection, PII redaction, and 
  [08] Datanous Embed supports text-embedding-3-small (1536 dims) and text-embedding-3-large (307
  [09] Redis caching in Datanous Embed achieves a cache hit rate above 60 percent on enterprise c
  [10] All Datanous.ai deployments use TLS 1.3 in transit and AES-256 at rest. GDPR-compliant.
  [11] A

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
 
# Step 1-3: Mocking the clustering process (Gaussian Mixture / UMAP)
# In a real scenario, you embed the corpus, reduce dimensionality with UMAP,
# and cluster with GMM. For this demo, we use a predefined mock dictionary.
clusters = {
    1: raptor_corpus[0:2] + raptor_corpus[14:15],  # Company Overview & Partners
    2: raptor_corpus[2:5],                         # Pricing tiers
    3: raptor_corpus[5:10],                        # Technical products (Search, Guard, Embed)
    4: raptor_corpus[10:14],                       # Case studies & Evaluation
}
print(f"Formed {len(clusters)} clusters from the corpus.\n")
 
# Step 4: Summarise each cluster into a higher-level (L2) summary node
 
CLUSTER_SUMMARY_PROMPT = ChatPromptTemplate.from_template(
    "Summarise the following related passages into ONE coherent paragraph "
    "that captures the main topic. Be concise and factually accurate.\n\n"
    "Passages:\n{texts}"
)
summarize_cluster = CLUSTER_SUMMARY_PROMPT | llm | StrOutputParser()
 
level_2_summaries = []
for cid, items in clusters.items():
    summary = summarize_cluster.invoke({"texts": "\n".join(items)})
    level_2_summaries.append(summary)
    print(f"[L2 Summary — Cluster {cid}]")
    print(summary)
    print()

Formed 4 clusters from the corpus.

[L2 Summary — Cluster 1]
Datanous.ai, an AI services company established in 2021 in Athens, Greece, employs 85 engineers and offers the Datanous Insight platform, which utilizes RAG technology to index enterprise documents with source citations. The company is also a certified partner of notable organizations such as OpenAI, Anthropic, Pinecone, and Weaviate.

[L2 Summary — Cluster 2]
Datanous offers three subscription plans for its Insight service: the Starter plan at $50 per month, which supports up to 10,000 document pages; the Professional plan at $350 per month, accommodating up to 100,000 pages; and the Enterprise plan, which provides unlimited pages along with a 99.9% uptime service level agreement (SLA).

[L2 Summary — Cluster 3]
Datanous.ai offers a suite of tools including Datanous Search, which integrates BM25 and dense embeddings through Reciprocal Rank Fusion with a latency under 200 ms, and Datanous Guard, which ensures factual groundin

In [9]:
# Step 5: Index both original leaves and L2 summaries
from langchain_chroma import Chroma
from langchain_core.documents import Document
 
raptor_docs = [Document(page_content=text, metadata={"level": "L1_leaf"}) for text in raptor_corpus]
raptor_docs += [Document(page_content=text, metadata={"level": "L2_summary"}) for text in level_2_summaries]
 
raptor_vectorstore = Chroma.from_documents(raptor_docs, embedder, collection_name="raptor_tree")
print(f"RAPTOR index built with {raptor_vectorstore._collection.count()} total nodes.")
 
# Test RAPTOR with a broad query
query = "What technical products does Datanous offer?"
results = raptor_vectorstore.similarity_search(query, k=2)
print(f"\nQuery: {query}\n")
for r in results:
    print(f"[{r.metadata['level']}] {r.page_content}")

RAPTOR index built with 19 total nodes.

Query: What technical products does Datanous offer?

[L2_summary] Datanous.ai, an AI services company established in 2021 in Athens, Greece, employs 85 engineers and offers the Datanous Insight platform, which utilizes RAG technology to index enterprise documents with source citations. The company is also a certified partner of notable organizations such as OpenAI, Anthropic, Pinecone, and Weaviate.
[L2_summary] Datanous.ai offers a suite of tools including Datanous Search, which integrates BM25 and dense embeddings through Reciprocal Rank Fusion with a latency under 200 ms, and Datanous Guard, which ensures factual grounding, prompt injection detection, PII redaction, and tenant isolation. Additionally, Datanous Embed supports two text embedding models with dimensions of 1536 and 3072, achieving over 60 percent cache hit rate on enterprise data through Redis caching. All deployments prioritize security with TLS 1.3 for data in transit and AES-2

💡 **Note:** Το πραγματικό RAPTOR χρησιμοποιεί Gaussian Mixture (UMAP για dimensionality reduction) και επαναλαμβάνει για πολλαπλά levels. Δες το επίσημο paper: arxiv.org/abs/2401.18059

## 5.4 Knowledge Graphs

Όταν τα δεδομένα σου είναι **εξαιρετικά δομημένα** (πχ entities + relationships), τα embeddings μπορεί να μην είναι η σωστή αναπαράσταση. Ένα **knowledge graph (KG)** μοντελοποιεί ρητά τα facts.

Παράδειγμα facts:


<img src="images/k-graph.png" height="700px" width="900px" style="border-radius:10px;margin:12px 0;"/>


Η ερώτηση «Ποια products της S-RAG έχουν SLA > 99.99%;» απαντιέται με **graph traversal**, όχι με embedding similarity.

**Hybrid:** Σε production συχνά συνδυάζονται embeddings (για unstructured search) + KG (για structured facts). Έχει αρχίσει να ονομάζεται **GraphRAG**.

💡 LangChain διαθέτει `LLMGraphTransformer` που εξάγει KG triples από κείμενο με LLM.

In [10]:
# Knowledge Graph extraction demo using LangChain's LLMGraphTransformer
# Note: requires `pip install langchain-experimental`
 
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer
# Not for production: OpenAI Structured Outputs + Pydantic + Neo4j driver, LlamaIndex PropertyGraphIndex κ.α.
 
print("LLMGraphTransformer uses an LLM to automatically extract nodes (entities) and edges (relationships).")
print("These can then be stored in a Graph Database like Neo4j for multi-hop retrieval (GraphRAG).\n")
 
# Από το κείμενο "text" θα προσπαθήσει να βρει ΟΝΤΟΤΗΤΕΣ όπως: Datanous.ai, Datanous Insight
# και ΣΧΕΣΕΙΣ όπως: Datanous.ai --[DEVELOPED]--> Datanous Insight
# Datanous Insight --[HAS_SLA]--> 99.9%
text = "Datanous.ai developed Datanous Insight. Datanous Insight has an SLA of 99.9%." #
print(f"Source Text: '{text}'\n")
 
graph_transformer = LLMGraphTransformer(llm=llm)
graph_docs = graph_transformer.convert_to_graph_documents([Document(page_content=text)])
 
for node in graph_docs[0].nodes:
    print(f"Node: {node.id} ({node.type})")
for edge in graph_docs[0].relationships:
    print(f"Edge: {edge.source.id} --[{edge.type}]--> {edge.target.id}")

LLMGraphTransformer uses an LLM to automatically extract nodes (entities) and edges (relationships).
These can then be stored in a Graph Database like Neo4j for multi-hop retrieval (GraphRAG).

Source Text: 'Datanous.ai developed Datanous Insight. Datanous Insight has an SLA of 99.9%.'

Node: Datanous.Ai (Organization)
Node: Datanous Insight (Product)
Node: 99.9% Sla (Service level agreement)
Edge: Datanous.Ai --[DEVELOPED]--> Datanous Insight
Edge: Datanous Insight --[HAS]--> 99.9% Sla


## 5.5 Συγκριτικός πίνακας

| Στρατηγική | Σενάριο | Πλεονεκτήματα | Μειονεκτήματα |
|---|---|---|---|
| **Naive chunking** | Μικρά docs, ομοιόμορφο corpus | Απλό, φθηνό | Trade-off context vs precision |
| **Multi-Representation** | Μεγάλα technical docs | Καθαρό embedding, πλήρες context στο LLM | +1 LLM call ανά doc στο indexing |
| **Parent Document** | Πολλά μέτρια docs | Καλό balance, χωρίς LLM στο indexing | Πιο πολύπλοκο storage |
| **RAPTOR** | Εκτενές corpus, mixed query types | Στηρίζει specific + broad queries | Ακριβό indexing (πολλά LLM calls) |
| **ColBERT** | Όταν χρειάζεσαι top-tier accuracy | Καλύτερη ποιότητα | Storage explosion, πιο αργό |
| **Knowledge Graph** | Δομημένα facts/entities | Επιτρέπει multi-hop reasoning | Πολύπλοκη κατασκευή |

## 5.6 Άσκηση

**Άσκηση:** Φτιάξε μια εκδοχή Multi-Representation που χρησιμοποιεί **3** αναπαραστάσεις:
1. Σύντομη σύνοψη (1 πρόταση)
2. Λίστα από πιθανές υποθετικές ερωτήσεις (3 ερωτήσεις)
3. Το full doc

Όλα τα παραπάνω indexάρονται ως δείκτες προς το ίδιο doc_id.

In [ ]:
# Exercise: Implement Multi-Representation retrieval for the Datanous.ai knowledge base.
#
# 1. Create a Chroma vector store and an InMemoryStore.
# 2. Initialize a MultiVectorRetriever.
# 3. For each document in datanous_docs:
#    a. Generate a 1-sentence summary.
#    b. Generate 3 hypothetical questions this document can answer.
#    c. Create Document objects for the summary and the questions, linking them to a common doc_id.
# 4. Add the summary/questions to the vector store, and the full doc to the doc store.
# 5. Test with a query and verify the original document is returned.
#
# Your solution here:


In [ ]:
# Advanced Indexing — techniques demonstrated in this notebook:
#
# 1. Parent-child retrieval  : index small chunks, return full parent documents
# 2. RAPTOR                  : hierarchical clustering + summarisation for multi-level retrieval
# 3. ColBERT                 : token-level late interaction for high-accuracy retrieval
#
# All techniques applied to the Datanous.ai knowledge base (datanous_docs).
print("Notebook 05 complete. See the cells above for each advanced indexing pattern.")


## 📋 Συμπεράσματα

| # | Έννοια | Συνοπτικά |
|---|---|---|
| 1 | Indexing dilemma | Μικρά chunks → καλό embed, μεγάλα → καλό context |
| 2 | Multi-Representation | Embed σύνοψη, return full doc |
| 3 | `MultiVectorRetriever` | Vector store για indices + doc store για originals |
| 4 | Parent Document | Embed children, return parents (auto split) |
| 5 | RAPTOR | Cluster leaves → summarize → δέντρο μεικτών επιπέδων |
| 6 | ColBERT | Token-level vectors + max-sim late interaction |
| 7 | Knowledge Graph | Δομημένα facts για multi-hop reasoning |
| 8 | GraphRAG | Hybrid embeddings + KG για structured + unstructured |
| 9 | Επόμενο βήμα | Notebook 06 — Re-ranking & Hybrid Search |